<!--html-->
<h1 style="text-align: center;">Домашняя работа по лекции 3</h1>
<strong style="display: block; text-align: center;">ФКН НИУ ВШЭ</strong>
<p style="text-align: center;">25 ноября 2024 г.</p>
<!--/html-->


## 1. Задание  

Рассмотрим механизм самовнимания в трансформерной модели.

1. Пусть заданы матрицы запросов $Q \in \mathbb{R}^{T \times d_k}$,  ключей $K \in \mathbb{R}^{T \times d_k}$ и значений $V \in \mathbb{R}^{T \times d_v}$. Выведите выражение для матрицы выхода внимания $Z \in \mathbb{R}^{T \times d_v}$, используя механизм самовнимания:
$$
   Z = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.
   $$
 Объясните пошагово, как вычисляются веса внимания и как масштабирующий фактор $\frac{1}{\sqrt{d_k}}$ влияет на стабильность градиентов при обучении.  

4. Опишите, как позиционные кодировки добавляются к входным эмбеддингам и почему они необходимы в трансформерных моделях, которые не используют рекуррентность.  

5. Для заданной последовательности длины $T$ сравните вычислительную сложность самовнимания в трансформерах с рекуррентными нейронными сетями $(RNN)$ и сетью долгосрочной краткосрочной памяти $(LSTM)$. Покажите, как масштабируется время вычислений и использование памяти в зависимости от $T$.  

6. Предложите модификацию стандартного механизма самовнимания для снижения вычислительной сложности при обработке очень длинных последовательностей. Опишите математически предложенный подход и обсудите его потенциальные преимущества и недостатки.

## Баллы

1. Задание — 2 балла  
2. Задание — 3 балла  
3. Задание — 2 балла  
4. Задание — 3 балла

# Решение задания №1

---

### 1. Пошаговые вычисления:

1. **Скалярное произведение запросов и ключей**:  
   Рассчитывается матрица внимания, основанная на скалярном произведении запросов $Q$ и ключей $K$:
   $$
   S = Q K^\top, \quad S \in \mathbb{R}^{T \times T}
   $$
   Здесь $S[i, j]$ определяет сходство между $i$-м запросом и $j$-м ключом.

2. **Масштабирование скалярного произведения**:  
   Для стабилизации градиентов результат нормируется на размерность $d_k$ через деление на $\sqrt{d_k}$:
   $$
   S_{\text{scaled}} = \frac{S}{\sqrt{d_k}} = \frac{Q K^\top}{\sqrt{d_k}}, \quad S_{\text{scaled}} \in \mathbb{R}^{T \times T}
   $$

3. **Применение softmax**:  
   К каждому строковому элементу матрицы $S_{\text{scaled}}$ применяется функция softmax, чтобы получить вероятности:
   $$
   A = \text{softmax}(S_{\text{scaled}}), \quad A \in \mathbb{R}^{T \times T}
   $$
   Элементы $A[i, j]$ интерпретируются как "веса внимания" $i$-го запроса к $j$-му ключу.

4. **Взвешенное суммирование значений**:  
   Итоговая матрица внимания $Z$ вычисляется через матричное умножение матрицы $A$ на значения $V$:
   $$
   Z = A V = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V, \quad Z \in \mathbb{R}^{T \times d_v}
   $$

---

### 2. Влияние масштабирующего фактора $\frac{1}{\sqrt{d_k}}$:

Масштабирующий фактор стабилизирует градиенты во время обучения:
1. Без нормализации значения $Q K^\top$ могут быть слишком большими, что приводит к резкому градиенту softmax, и модель плохо обучается.
2. Деление на $\sqrt{d_k}$ уменьшает амплитуду значений, снижая вероятность возникновения затухающих или взрывающихся градиентов.

---



# Решение задания №2

---

Трансформерные модели, такие как архитектура **Transformer**, не содержат рекуррентных или сверточных слоев, поэтому они не имеют встроенной способности обрабатывать последовательность данных по порядку. Чтобы модель могла учитывать относительное и абсолютное положение элементов в последовательности, вводятся **позиционные кодировки (Positional Encodings)**.

---

### Как добавляются позиционные кодировки:

1. **Эмбеддинг входных токенов:**  
   Каждый входной токен (слово, символ и т.д.) представляется как эмбеддинг фиксированной размерности. Пусть эмбеддинги имеют размерность $d_{\text{model}}$.

2. **Генерация позиционных кодировок:**  
   Используются обучаемые векторы для кодирования позиций. Для каждого токена в последовательности генерируется обучаемый вектор $\text{Emb}_{\text{pos}}(i)$, где $i$ — позиция токена. Эти векторы обучаются вместе с другими параметрами модели.

3. **Сложение с эмбеддингами:**  
   Для каждого токена его эмбеддинг $E$ и соответствующая позиционная кодировка $\text{Emb}_{\text{pos}}$ складываются:
   $$
   E_{\text{input}} = E + \text{Emb}_{\text{pos}}
   $$
   Вектор $E_{\text{input}}$ поступает на вход модели трансформера.

---

### Почему они необходимы:

1. **Учет порядка в последовательности:**  
   Без позиционных кодировок трансформер рассматривает токены как независимые объекты, не различая их положение в последовательности. Позиционные кодировки добавляют информацию о положении каждого токена.Это необходимо для того, чтобы видеть разницу между "очень хорошо, совсем не плохо" и "не очень хорошо, совсем плохо"

2. **Обучаемые эмбеддинги:**  
   Обучаемые позиционные эмбеддинги дают модели возможность самостоятельно оптимизировать представления позиций токенов, что может быть более эффективно для специфических задач.

3. **Замена рекуррентности:**  
   В рекуррентных моделях порядок токенов обрабатывается итеративно. В трансформерах этот порядок задается явно через позиционные кодировки, что позволяет модели обрабатывать последовательность параллельно.

---




# Решение задания №3

---

### Сравнение вычислительной сложности

| Модель            | Вычислительная сложность      | Объяснение                                                                                   |
|--------------------|-------------------------------|---------------------------------------------------------------------------------------------|
| **RNN**           | $\mathcal{O}(T \cdot d^2)$   | RNN обрабатывает последовательность итеративно, шаг за шагом, обновляя скрытое состояние.   |
| **LSTM**          | $\mathcal{O}(T \cdot d^2)$   | LSTM улучшает RNN с помощью дополнительных векторов состояния (ячейки памяти), но сложность остается той же. |
| **Self-Attention**| $\mathcal{O}(T^2 \cdot d)$   | Самовнимание вычисляет все связи между токенами в последовательности одновременно, что требует матрицы внимания размером $T \times T$. |

- $T$ — длина последовательности.  
- $d$ — размерность эмбеддингов.

---

### Масштабируемость

#### 1. **Время вычислений**
- В RNN и LSTM каждая операция зависит от результата предыдущей (итеративный процесс). Время вычислений растет линейно с $T$:  
  $\mathcal{O}(T)$.
- В Self-Attention все токены обрабатываются параллельно, но вычисление внимания требует обработки всех пар токенов. Время растет квадратично:  
  $\mathcal{O}(T^2)$.

#### 2. **Использование памяти**
- В RNN и LSTM требуется хранить только текущее скрытое состояние (плюс дополнительные состояния для LSTM). Память масштабируется как:  
  $\mathcal{O}(T \cdot d)$.
- В Self-Attention нужно хранить матрицу внимания размером $T \times T$, что требует квадратичной памяти:  
  $\mathcal{O}(T^2 \cdot d)$.

---



### Итог

- Для **коротких последовательностей** Self-Attention выгоднее, так как он обрабатывает данные параллельно и лучше учитывает глобальный контекст.
- Для **длинных последовательностей** RNN и LSTM могут быть более подходящими, если вычислительные ресурсы ограничены.

